In [1]:
import requests
import json
import numpy as np

# ── Configuration ──
FDA_ONLY = True  # Set to False to include all drugs (not just FDA-approved)

# Build query based on FDA filter
if FDA_ONLY:
    query = {
        "query": {
            "type": "terminal",
            "service": "text_chem",
            "parameters": {
                "attribute": "drugbank_info.drug_groups",
                "operator": "exact_match",
                "value": "approved",
                "negation": False
            }
        },
        "return_type": "mol_definition",
        "request_options": {
            "paginate": {"start": 0, "rows": 3000},
            "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
        }
    }
else:
    query = {
        "query": {
            "type": "terminal",
            "service": "text_chem",
            "parameters": {
                "attribute": "drugbank_container_identifiers.drugbank_id",
                "operator": "exists",
                "negation": False
            }
        },
        "return_type": "mol_definition",
        "request_options": {
            "paginate": {"start": 0, "rows": 3000},
            "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
        }
    }

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()

comp_ids = [r["identifier"] for r in results.get("result_set", [])]
label = "FDA-approved" if FDA_ONLY else "all DrugBank"
print(f"Found {len(comp_ids)} {label} drugs")
print("First 10:", comp_ids[:10])

Found 988 FDA-approved drugs
First 10: ['B1Z', 'PRD_900028', 'PRD_000204', 'BLM', 'CNC', 'COB', 'IDB', 'QWP', 'FI8', 'A4I']


In [ ]:
import requests
import os
import time
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
from scipy.spatial.distance import pdist

# Suppress noisy RDKit warnings (2D/3D tagging, valence issues)
RDLogger.DisableLog('rdApp.*')

data_dir = os.path.expanduser("~/workspace/storage")

os.makedirs(os.path.join(data_dir, "ligands_sdf_large"), exist_ok=True)

# ── Configuration ──
FDA_ONLY = False             # Set to False to include all DrugBank drugs
REQUIRE_COMPLEX = False      # True = large AND complex; False = large only
NUM_LIGANDS = 525           # Number of largest drugs to keep
MIN_MW = 100                # Minimum formula weight (Da)
MIN_HEAVY_ATOMS = 15        # Minimum heavy atoms for "large"
MIN_EXTENT = 15.0           # Minimum spatial extent (Å)
MIN_ROTATABLE_BONDS = 3     # Minimum rotatable bonds for "complex" (only if REQUIRE_COMPLEX)
MIN_RINGS = 2              # Minimum ring count for "complex" (only if REQUIRE_COMPLEX)

# Step 1: Query RCSB for drugs with high MW
fda_node = {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_info.drug_groups",
        "operator": "exact_match",
        "value": "approved",
        "negation": False
    }
} if FDA_ONLY else {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_container_identifiers.drugbank_id",
        "operator": "exists",
        "negation": False
    }
}

label = "FDA-approved" if FDA_ONLY else "all DrugBank"
mode = "large & complex" if REQUIRE_COMPLEX else "large only"
print(f"Querying RCSB for {label} drugs (mode: {mode})...")

query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            fda_node,
            {
                "type": "terminal",
                "service": "text_chem",
                "parameters": {
                    "attribute": "chem_comp.formula_weight",
                    "operator": "greater",
                    "value": MIN_MW,
                    "negation": False
                }
            }
        ]
    },
    "return_type": "mol_definition",
    "request_options": {
        "paginate": {"start": 0, "rows": 5000},
        "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "desc"}]
    }
}

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()
fda_ids = [r["identifier"] for r in results.get("result_set", [])]
print(f"Found {len(fda_ids)} {label} drugs with MW > {MIN_MW} Da")

# Step 2: Download SDF, compute size & complexity, filter
def compute_extent_from_sdf(filepath):
    """Max atom-to-atom distance (Å) from SDF coordinates."""
    coords = []
    with open(filepath) as f:
        lines = f.readlines()
    try:
        num_atoms = int(lines[3][:3])
        for i in range(4, 4 + num_atoms):
            parts = lines[i].split()
            coords.append([float(parts[0]), float(parts[1]), float(parts[2])])
    except (ValueError, IndexError):
        return 0.0
    if len(coords) < 2:
        return 0.0
    return pdist(np.array(coords)).max()

failed = []
filtered_drugs = []  # (comp_id, extent, mw, n_heavy, n_rot, n_rings, smiles)
cached = 0

for i, cid in enumerate(fda_ids):
    filepath = os.path.join(data_dir, "ligands_sdf_large", f"{cid}_ideal.sdf")

    # Skip download if file already exists and is non-empty
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        cached += 1
    else:
        dl_url = f"https://files.rcsb.org/ligands/download/{cid}_ideal.sdf"
        r = requests.get(dl_url)
        if r.status_code != 200 or "V2000" not in r.text:
            failed.append(cid)
            continue

        with open(filepath, "w") as f:
            f.write(r.text)

    # Parse with RDKit
    mol = Chem.MolFromMolFile(filepath, sanitize=True, removeHs=True)
    if mol is None:
        failed.append(cid)
        continue

    n_heavy = mol.GetNumHeavyAtoms()
    n_rot = Descriptors.NumRotatableBonds(mol)
    n_rings = rdMolDescriptors.CalcNumRings(mol)
    mw = Descriptors.MolWt(mol)
    extent = compute_extent_from_sdf(filepath)

    # Size filter (always applied)
    is_large = (n_heavy >= MIN_HEAVY_ATOMS and extent >= MIN_EXTENT)

    # Complexity filter (only when REQUIRE_COMPLEX is True)
    is_complex = (n_rot >= MIN_ROTATABLE_BONDS and n_rings >= MIN_RINGS)

    if REQUIRE_COMPLEX:
        if is_large and is_complex:
            smiles = Chem.MolToSmiles(mol)
            filtered_drugs.append((cid, extent, mw, n_heavy, n_rot, n_rings, smiles))
    else:
        if is_large:
            smiles = Chem.MolToSmiles(mol)
            filtered_drugs.append((cid, extent, mw, n_heavy, n_rot, n_rings, smiles))

    print(f"Processed {i}/{len(fda_ids)} — {len(filtered_drugs)} {mode} drugs found... ({cached} cached)")
    if not (os.path.exists(filepath) and os.path.getsize(filepath) > 0):
        time.sleep(0.05)

# Sort by extent descending and keep top N largest
filtered_drugs.sort(key=lambda x: -x[1])
large_drugs = filtered_drugs[:NUM_LIGANDS]

print(f"\nDone. Failed downloads: {len(failed)} | Cached (skipped download): {cached}")
print(f"{mode.capitalize()} {label} drugs found: {len(filtered_drugs)}")
print(f"Keeping top {NUM_LIGANDS} largest: {len(large_drugs)} drugs")
if large_drugs:
    print(f"Size range: {large_drugs[-1][1]} – {large_drugs[0][1]} Å")
print(f"\nTop 30 largest {mode} {label} drugs:")
print(f"{'ID':>8}  {'Å':>6}  {'MW':>7}  {'Atoms':>5}  {'Rot':>3}  {'Ring':>4}  SMILES")
print("-" * 80)
for cid, ext, mw, nh, nr, nring, smi in large_drugs[:30]:
    print(f"{cid:>8}  {ext:>6.1f}  {mw:>7.1f}  {nh:>5}  {nr:>3}  {nring:>4}  {smi[:40]}...")

Querying RCSB for all DrugBank drugs (mode: large only)...
Found 5000 all DrugBank drugs with MW > 100 Da


FileNotFoundError: [Errno 2] No such file or directory: 'ligands_sdf_large/TBR_ideal.sdf'

In [ ]:
# --- Copy approved ligands (RDKit-readable) to a separate folder ---
import glob
import shutil
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

APPROVED_DIR = os.path.join(data_dir, "ligands_sdf_large_approved")
os.makedirs(APPROVED_DIR, exist_ok=True)

# large_drugs is already sorted by extent descending and sliced to NUM_LIGANDS
kept_ids = {cid for cid, *_ in large_drugs}

sdf_files = glob.glob(os.path.join(data_dir, "ligands_sdf_large", "*.sdf"))
copied = 0
skipped_not_kept = 0
skipped_unreadable = 0

for filepath in sdf_files:
    filename = os.path.basename(filepath)
    cid = filename.replace("_ideal.sdf", "")

    # Must be in the top NUM_LIGANDS by extent
    if cid not in kept_ids:
        skipped_not_kept += 1
        continue

    # Must be readable by RDKit (ligand_props_valid equivalent)
    try:
        supplier = Chem.SDMolSupplier(filepath, sanitize=False, removeHs=False)
        mol = next(iter(supplier), None)
        if mol is None:
            skipped_unreadable += 1
            continue
        Chem.SanitizeMol(mol)
        if mol.GetNumHeavyAtoms() is None or mol.GetNumHeavyAtoms() == 0:
            skipped_unreadable += 1
            continue
    except Exception:
        skipped_unreadable += 1
        continue

    shutil.copy2(filepath, os.path.join(APPROVED_DIR, filename))
    copied += 1

# Summary
extents = {cid: ext for cid, ext, *_ in large_drugs}
approved_files = glob.glob(f"{APPROVED_DIR}/*.sdf")
kept_extents = sorted(
    [extents[os.path.basename(f).replace("_ideal.sdf", "")]
     for f in approved_files
     if os.path.basename(f).replace("_ideal.sdf", "") in extents],
    reverse=True
)

print(f"Copied to {APPROVED_DIR}/: {copied} approved ligands")
print(f"Skipped (not in top {NUM_LIGANDS}): {skipped_not_kept}")
print(f"Skipped (RDKit unreadable)       : {skipped_unreadable}")
print(f"Original ligands_sdf_large/      : {len(sdf_files)} files (unchanged)")
if kept_extents:
    print(f"Extent range kept: {kept_extents[-1]:.1f} Å – {kept_extents[0]:.1f} Å")
    print(f"Median extent    : {kept_extents[len(kept_extents)//2]:.1f} Å")
print(f"\nTop 10 largest approved ligands:")
print(f"{'Rank':>4}  {'ID':>8}  {'Extent (Å)':>10}  {'MW':>7}  {'Atoms':>5}")
print("-" * 45)
for rank, (cid, ext, mw, nh, *_) in enumerate(large_drugs[:10], 1):
    print(f"{rank:>4}  {cid:>8}  {ext:>10.1f}  {mw:>7.1f}  {nh:>5}")

Copied to /root/workspace/storage/ligands_sdf_large_approved/: 0 approved ligands
Skipped (not in top 525): 4957
Skipped (RDKit unreadable)       : 0
Original ligands_sdf_large/      : 4957 files (unchanged)

Top 10 largest approved ligands:
Rank        ID  Extent (Å)       MW  Atoms
---------------------------------------------


In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import os
import time
import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors

# Suppress noisy RDKit warnings
RDLogger.DisableLog('rdApp.*')

os.makedirs(os.path.join(data_dir, "ligands_sdf_small_symmetric"), exist_ok=True)

# ── Configuration ──
FDA_ONLY = False             # Set to False to include all DrugBank drugs
NUM_LIGANDS = 550           # Number of small symmetric ligands to keep
MAX_MW = 500                # Maximum formula weight (Da) — defines "small"
MIN_HEAVY_ATOMS = 3         # Minimum heavy atoms (exclude trivial molecules)
MAX_HEAVY_ATOMS = 25        # Maximum heavy atoms — enforces "small"
MAX_ROTATABLE_BONDS = 4     # Maximum rotatable bonds — enforces "primitive/rigid"
MAX_RINGS = 3               # Maximum ring count — enforces "primitive"
MAX_SYMMETRY_RATIO = 0.85    # Lower = more symmetric (unique environments / total heavy atoms)
                            # Benzene = 0.17, Aspirin ≈ 1.0, Naphthalene = 0.2
MAX_CHIRAL_CENTERS = 2      # Max stereocenters — 0 = fully achiral (more symmetric)

# Step 1: Query RCSB for small drugs
fda_node = {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_info.drug_groups",
        "operator": "exact_match",
        "value": "approved",
        "negation": False
    }
} if FDA_ONLY else {
    "type": "terminal",
    "service": "text_chem",
    "parameters": {
        "attribute": "drugbank_container_identifiers.drugbank_id",
        "operator": "exists",
        "negation": False
    }
}

label = "FDA-approved" if FDA_ONLY else "all DrugBank"
print(f"Querying RCSB for small, primitive & symmetric {label} drugs...")
print(f"  MW ≤ {MAX_MW} Da | Atoms {MIN_HEAVY_ATOMS}–{MAX_HEAVY_ATOMS} | "
      f"Rot ≤ {MAX_ROTATABLE_BONDS} | Rings ≤ {MAX_RINGS} | "
      f"Sym ratio ≤ {MAX_SYMMETRY_RATIO} | Chiral ≤ {MAX_CHIRAL_CENTERS}")

query = {
    "query": {
        "type": "group",
        "logical_operator": "and",
        "nodes": [
            fda_node,
            {
                "type": "terminal",
                "service": "text_chem",
                "parameters": {
                    "attribute": "chem_comp.formula_weight",
                    "operator": "less",
                    "value": MAX_MW,
                    "negation": False
                }
            }
        ]
    },
    "return_type": "mol_definition",
    "request_options": {
        "paginate": {"start": 0, "rows": 10000},
        "sort": [{"sort_by": "chem_comp.formula_weight", "direction": "asc"}]
    }
}

url = "https://search.rcsb.org/rcsbsearch/v2/query"
response = requests.post(url, json=query)
results = response.json()
candidate_ids = [r["identifier"] for r in results.get("result_set", [])]
print(f"Found {len(candidate_ids)} {label} drugs with MW < {MAX_MW} Da")

# Step 2: Download SDF, parse with RDKit, filter for small + primitive + symmetric
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[429, 500, 502, 503, 504],
                allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=10))

def compute_symmetry_ratio(mol):
    """Ratio of unique atomic environments to total heavy atoms.
    Lower = more symmetric. Benzene=0.17, fully asymmetric=1.0."""
    n_heavy = mol.GetNumHeavyAtoms()
    if n_heavy == 0:
        return 1.0
    ranks = Chem.CanonicalRankAtoms(mol, breakTies=False)
    n_unique = len(set(ranks))
    return n_unique / n_heavy

failed = []
filtered_ligands = []  # (comp_id, sym_ratio, mw, n_heavy, n_rot, n_rings, n_chiral, smiles)
cached = 0

for i, cid in enumerate(candidate_ids):
    filepath = os.path.join(data_dir, "ligands_sdf_small_symmetric", f"{cid}_ideal.sdf")

    # Skip download if file already exists and is non-empty
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        cached += 1
    else:
        dl_url = f"https://files.rcsb.org/ligands/download/{cid}_ideal.sdf"
        try:
            r = session.get(dl_url, timeout=15)
        except requests.exceptions.RequestException:
            failed.append(cid)
            continue

        if r.status_code != 200 or "V2000" not in r.text:
            failed.append(cid)
            continue

        with open(filepath, "w") as f:
            f.write(r.text)

    # Parse with RDKit
    mol = Chem.MolFromMolFile(filepath, sanitize=True, removeHs=True)
    if mol is None:
        failed.append(cid)
        continue

    n_heavy = mol.GetNumHeavyAtoms()
    mw = Descriptors.MolWt(mol)
    n_rot = Descriptors.NumRotatableBonds(mol)
    n_rings = rdMolDescriptors.CalcNumRings(mol)
    n_chiral = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))

    # Filter: small
    if n_heavy < MIN_HEAVY_ATOMS or n_heavy > MAX_HEAVY_ATOMS:
        continue

    # Filter: primitive (rigid, few rings)
    if n_rot > MAX_ROTATABLE_BONDS:
        continue
    if n_rings > MAX_RINGS:
        continue

    # Filter: symmetric (low unique-environment ratio, achiral)
    sym_ratio = compute_symmetry_ratio(mol)
    if sym_ratio > MAX_SYMMETRY_RATIO:
        continue
    if n_chiral > MAX_CHIRAL_CENTERS:
        continue

    smiles = Chem.MolToSmiles(mol)
    filtered_ligands.append((cid, sym_ratio, mw, n_heavy, n_rot, n_rings, n_chiral, smiles))

    print(f"Processed {i}/{len(candidate_ids)} — {len(filtered_ligands)} small symmetric primitives found... ({cached} cached)")
    if not (os.path.exists(filepath) and os.path.getsize(filepath) > 0):
        time.sleep(0.15)

# Sort by symmetry ratio (most symmetric first), then by fewest atoms
filtered_ligands.sort(key=lambda x: (x[1], x[3]))
kept_ligands = filtered_ligands[:NUM_LIGANDS]

print(f"\nDone. Failed downloads: {len(failed)} | Cached (skipped download): {cached}")
print(f"Small, primitive & symmetric {label} drugs found: {len(filtered_ligands)}")
print(f"Keeping top {NUM_LIGANDS}: {len(kept_ligands)} drugs")
if kept_ligands:
    print(f"Symmetry ratio range: {kept_ligands[0][1]} – {kept_ligands[-1][1]}")

print(f"\nTop 30 most symmetric small primitives:")
print(f"{'ID':>8}  {'Sym':>5}  {'MW':>7}  {'Atoms':>5}  {'Rot':>3}  {'Ring':>4}  {'Chir':>4}  SMILES")
print("-" * 85)
for cid, sym, mw, nh, nr, nring, nchi, smi in kept_ligands[:30]:
    print(f"{cid:>8}  {sym:>5.3f}  {mw:>7.1f}  {nh:>5}  {nr:>3}  {nring:>4}  {nchi:>4}  {smi}")

IndentationError: unexpected indent (2907654287.py, line 156)

In [ ]:
# --- Copy approved small/symmetric ligands (RDKit-readable) to a separate folder ---
import glob
import shutil
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

APPROVED_SMALL_DIR = os.path.join(data_dir, "ligands_sdf_small_symmetric_approved")
os.makedirs(APPROVED_SMALL_DIR, exist_ok=True)

kept_ids = {cid for cid, *_ in kept_ligands}
sdf_files = glob.glob(os.path.join(data_dir, "ligands_sdf_small_symmetric", "*.sdf"))
copied = 0
skipped_not_kept = 0
skipped_unreadable = 0

for filepath in sdf_files:
    filename = os.path.basename(filepath)
    cid = filename.replace("_ideal.sdf", "")

    if cid not in kept_ids:
        skipped_not_kept += 1
        continue

    try:
        supplier = Chem.SDMolSupplier(filepath, sanitize=False, removeHs=False)
        mol = next(iter(supplier), None)
        if mol is None:
            skipped_unreadable += 1
            continue
        Chem.SanitizeMol(mol)
        if mol.GetNumHeavyAtoms() is None or mol.GetNumHeavyAtoms() == 0:
            skipped_unreadable += 1
            continue
    except Exception:
        skipped_unreadable += 1
        continue

    shutil.copy2(filepath, os.path.join(APPROVED_SMALL_DIR, filename))
    copied += 1

# Summary
sym_ratios_map = {cid: sym for cid, sym, *_ in kept_ligands}
approved_files = glob.glob(f"{APPROVED_SMALL_DIR}/*.sdf")
kept_sym_ratios = sorted(
    [sym_ratios_map[os.path.basename(f).replace("_ideal.sdf", "")]
     for f in approved_files
     if os.path.basename(f).replace("_ideal.sdf", "") in sym_ratios_map],
)

remaining_orig = len(sdf_files)
print(f"Copied to {APPROVED_SMALL_DIR}/: {copied} approved ligands")
print(f"Skipped (not in kept list)  : {skipped_not_kept}")
print(f"Skipped (RDKit unreadable)  : {skipped_unreadable}")
print(f"Original ligands_sdf_small_symmetric/: {remaining_orig} files (unchanged)")
if kept_sym_ratios:
    print(f"Symmetry ratio range kept: {kept_sym_ratios[0]:.3f} – {kept_sym_ratios[-1]:.3f}")
    print(f"Median symmetry ratio    : {kept_sym_ratios[len(kept_sym_ratios)//2]:.3f}")
print(f"\nTop 10 most symmetric approved ligands:")
print(f"{'Rank':>4}  {'ID':>8}  {'Sym':>5}  {'MW':>7}  {'Atoms':>5}")
print("-" * 40)
for rank, (cid, sym, mw, nh, *_) in enumerate(kept_ligands[:10], 1):
    print(f"{rank:>4}  {cid:>8}  {sym:>5.3f}  {mw:>7.1f}  {nh:>5}")

Removed 5294 SDF files that didn't pass filters
Remaining in ligands_sdf_small_symmetric/: 550 files
